In [1]:
from datetime import datetime
import MetaTrader5 as mt5
import pandas as pd
import pytz
import time
import pandas_ta as pta
import matplotlib.pyplot as plt 
# import threading
mt5.initialize()


True

In [2]:
def price_action(symbol, lot, ask, bid, order_type):
    buy_profit=mt5.order_calc_profit(order_type,symbol,lot,ask,bid)
    return buy_profit
price_action("EURUSD", 0.02, 1.18969, 1.17030,mt5.ORDER_TYPE_SELL)

38.78

In [9]:
def get_values(symbol):
    rates = mt5.copy_rates_from_pos(symbol, mt5.TIMEFRAME_H1, 0, 2000)
    rates_frame = pd.DataFrame(rates)

    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    rates_frame['rsi'] = get_rsi(rates_frame['close'], 24)
    
    rates_frame['sma1']= rates_frame['close'].rolling(window=10).mean()
    rates_frame['sma2']= rates_frame['close'].rolling(window=13).mean()

    return rates_frame

In [10]:
def get_rsi(close, lookback):
#     t = time.time()
    ret = close.diff()
    
    up = []
    down = []
    for i in range(len(ret)):
        if ret[i] < 0:
            up.append(0)
            down.append(ret[i])
        else:
            up.append(ret[i])
            down.append(0)
    up_series = pd.Series(up)
    down_series = pd.Series(down).abs()
    up_ewm = up_series.ewm(com = lookback - 1, adjust = False).mean()
    down_ewm = down_series.ewm(com = lookback - 1, adjust = False).mean()
    rs = up_ewm/down_ewm
    rsi = 100 - (100 / (1 + rs))
    rsi_df = pd.DataFrame(rsi).rename(columns = {0:'rsi'}).set_index(close.index)
#     print(time.time()-t)
    return rsi_df

In [11]:
symbol = "EURUSD"
a = get_values(symbol)
a

,open,high,low,close,tick_volume,spread,real_volume,rsi,sma1,sma2
time,,,,,,,,,,
2021-05-14 13:00:00,1.21097,1.21153,1.21027,1.21136,2812,6,0,NaN,NaN,NaN
2021-05-14 14:00:00,1.21137,1.21257,1.21101,1.21192,3456,6,0,100.000000,NaN,NaN
2021-05-14 15:00:00,1.21193,1.21367,1.21162,1.21293,5919,6,0,100.000000,NaN,NaN
2021-05-14 16:00:00,1.21294,1.21446,1.21240,1.21406,5419,6,0,100.000000,NaN,NaN
2021-05-14 17:00:00,1.21406,1.21474,1.21315,1.21340,5607,6,0,95.448129,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
2021-09-08 16:00:00,1.18207,1.18222,1.18084,1.18170,4645,6,0,34.351897,1.182566,1.183004
2021-09-08 17:00:00,1.18170,1.18190,1.18018,1.18120,5784,6,0,32.613570,1.182253,1.182732
2021-09-08 18:00:00,1.18119,1.18208,1.18107,1.18180,3729,6,0,36.629035,1.182069,1.182527


In [43]:
# less negatives more positives

B = []
check = 0
profit = []
index = []
indexB = []
counter = 0
peck = 0
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000

lenn = len(a)

for i in range(20, len(a)):
    if a.iloc[i].sma1 > a.iloc[i].sma2 and a.iloc[i].rsi >= 49.0:
        buy_price = a.iloc[i].close
        print("#"*20)
        print(a.iloc[i].name)
        print("*"*20)
        check = 1  
        up = 0
        k = 0.0

    elif check == 1:
        sell_price = a.iloc[i].high
        pp = price_action(symbol, 0.02, buy_price, sell_price,mt5.ORDER_TYPE_BUY)
        print(f"{pp}---{a.iloc[i].high}--{a.iloc[i].name}")
        profit.append(pp)
        check = 0
        sell_price = a.iloc[i].low
        pp = price_action(symbol, 0.02, buy_price, sell_price,mt5.ORDER_TYPE_SELL)
        print(f"{pp}---{a.iloc[i].low}--{a.iloc[i].name}")
        profit.append(pp)
        check = 0

####################
2021-08-03 06:00:00
********************
1.25---109.211--2021-08-03 12:00:00
5.0---108.871--2021-08-03 12:00:00
####################
2021-08-03 18:00:00
********************
1.94---109.105--2021-08-04 00:00:00
1.45---108.92--2021-08-04 00:00:00
####################
2021-08-04 06:00:00
********************
7.9399999999999995---109.621--2021-08-04 12:00:00
8.63---108.717--2021-08-04 12:00:00
####################
2021-08-04 18:00:00
********************
4.59---109.713--2021-08-05 00:00:00
0.38---109.44--2021-08-05 00:00:00
####################
2021-08-05 06:00:00
********************
3.24---109.773--2021-08-05 12:00:00
3.6---109.398--2021-08-05 12:00:00
####################
2021-08-05 18:00:00
********************
2.93---109.883--2021-08-06 00:00:00
-0.27---109.737--2021-08-06 00:00:00
####################
2021-08-06 06:00:00
********************
10.42---110.351--2021-08-06 12:00:00
0.47---109.75--2021-08-06 12:00:00
####################
2021-08-06 18:00:00
**********

In [44]:
n = 0
p = 0
tn = 0
tp = 0

for i in profit:
    if i<0.0:
        n = n+i
        tn = tn+1
    else:
        p = p+i
        tp = tp+1
print(f"Total negative sm -->{n}")
print(f"Total negative -->{tn}")      
print(f"Total positive sm -->{p}")      
print(f"Total positive -->{tp}") 
print(f"Length {len(profit)}")

Total negative sm -->-1.07
Total negative -->3
Total positive sm -->188.03000000000006
Total positive -->65
Length 68


In [ ]:
#         if a.iloc[i-3].rsi < a.iloc[i-2].rsi and a.iloc[i-1].rsi < a.iloc[i-2].rsi \
#             and a.iloc[i-1].rsi < a.iloc[i].rsi and a.iloc[i].rsi < a.iloc[i-2].rsi \
#             and a.iloc[i].close > a.iloc[i].open \
#             and (a.iloc[i].rsi - a.iloc[i-1].rsi) >= 2.0 \
#              and check == 0:
#             and (a.iloc[i-2].rsi - a.iloc[i-3].rsi) >= 2.0

In [24]:
#less negatives more positives

B = []
check = 0
profit = []
index = []
indexB = []
counter = 0
peck = 0
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000

for i in range(5, len(a)):
    if a.iloc[i-1].rsi < a.iloc[i].rsi and (a.iloc[i].rsi - a.iloc[i-1].rsi) >= 2.0 \
        and a.iloc[i-2].rsi > a.iloc[i-1].rsi \
        and check == 0:

        buy_price = a.iloc[i+1].open
        print("#"*20)
        print(a.iloc[i+1].name)
        print("*"*20)
        check = 1  
        up = 0
        dp = 0
        k = 0.0

    elif check == 1:
        sell_price = a.iloc[i].close
        pp = price_action(symbol, 0.02, buy_price, sell_price,mt5.ORDER_TYPE_SELL)
        print(f"{pp}---{round(a.iloc[i].rsi, 2)}---{a.iloc[i].close}--{a.iloc[i].name}")

#         if pp >= 0.02:
#             profit.append(pp)
#             check = 0
        if pp > 0.0:
            up = up+1
            if up > 1: 
                profit.append(pp)
                check = 0
            dp = 1
#         elif pp < 0.0:
#             dp = dp+1
#             if dp > 1: 
#                 profit.append(pp)
#                 check = 0
#             up = 0

#             elif pp < -2.0:
#                 profit.append(pp)
#                 check = 0

        elif pp > 0.0:
            up = up+1
            if up == 1:
                k = pp
            if up > 1:
                if pp > k:
                    print("pass")
                else:
                    profit.append(pp)
                    check = 0
            k = pp
                #In live trade if loss still goes on to increase then close the trade before hand

####################
2021-08-10 06:00:00
********************
1.22---4.97---1.173--2021-08-10 06:00:00
3.8---4.11---1.17171--2021-08-10 12:00:00
####################
2021-08-11 06:00:00
********************
2.5---14.08---1.17104--2021-08-11 06:00:00
-2.82---35.53---1.1737--2021-08-11 12:00:00
-2.14---34.35---1.17336--2021-08-11 18:00:00
-2.9---36.88---1.17374--2021-08-12 00:00:00
-4.46---41.82---1.17452--2021-08-12 06:00:00
-0.98---35.2---1.17278--2021-08-12 12:00:00
-0.9---35.06---1.1727400000000001--2021-08-12 18:00:00
-3.24---42.17---1.17391--2021-08-13 00:00:00
-3.56---43.09---1.17407--2021-08-13 06:00:00
-13.88---63.32---1.17923--2021-08-13 12:00:00
-13.76---63.04---1.17917--2021-08-13 18:00:00
-14.86---64.59---1.17972--2021-08-16 00:00:00
-12.86---59.69---1.17872--2021-08-16 06:00:00
-12.82---59.59---1.1787--2021-08-16 12:00:00
-9.94---52.9---1.17726--2021-08-16 18:00:00
-9.84---52.68---1.17721--2021-08-17 00:00:00
-10.72---54.49---1.17765--2021-08-17 06:00:00
1.22---35.0---1.171

In [25]:
n = 0
p = 0
tn = 0
tp = 0
print(sum(profit))
for i in profit:
    if i<0.0:
        n = n+i
        tn = tn+1
    else:
        p = p+i
        tp = tp+1
print(f"Total negative sm -->{n}")
print(f"Total negative -->{tn}")      
print(f"Total positive sm -->{p}")      
print(f"Total positive -->{tp}") 
print(f"Length {len(profit)}")

10.18
Total negative sm -->0
Total negative -->0
Total positive sm -->10.18
Total positive -->5
Length 5
